# BraTS 2024 - Przygotowanie i Eksport Zbioru Wycinków (2D Slices)

Ten notebook odpowiada za:
1. Montowanie Google Drive i instalację pakietu `brain_mri`.
2. Wczytanie konfiguracji preprocessingu i podziału pacjentów (`train` / `validate` / `test`).
3. **Test na 1 pacjencie:** weryfikację poprawności wycinków, tensorów PyTorcha oraz wizualizację.
4. **Pełny eksport zbioru:** zapis gotowych wycinków do plików `.pt` na dysku.

## 1. Montowanie Google Drive i konfiguracja środowiska

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
from pathlib import Path

repo_path = Path("/content/BraTS_thesis")
branch = "minvalue_norm"

if not repo_path.exists():
    !git clone -b {branch} https://github.com/AdaBro24/BraTS_thesis.git {repo_path}
else:
    !git -C {repo_path} fetch origin
    !git -C {repo_path} checkout {branch}
    !git -C {repo_path} pull origin {branch}

!pip install -e /content/BraTS_thesis

## 2. Ścieżki danych i wczytanie konfiguracji

In [ ]:
from pathlib import Path
import torch

from brain_mri import load_data_config
from brain_mri.data.split import get_or_create_splits
from brain_mri.data.slice_extraction import extract_patient_slices
from brain_mri.data.dataset_export import save_patients_to_pt, export_dataset_split

# Ścieżki danych na Google Drive
ZIP_DIR = Path("/content/drive/MyDrive/engineering_thesis/brats")
DATA_ROOT = ZIP_DIR / "unpacked"
GLI_ROOT = DATA_ROOT / "BraTS-GLI" / "training_data1_v2"

# Wczytanie konfiguracji YAML z repozytorium
config = load_data_config(Path("/content/BraTS_thesis/configs/data_config.yaml"))
# Uaktualnienie ścieżki do granic Z na pełną ścieżkę w Colabie
config.paths.z_boundaries = Path("/content/BraTS_thesis/data/patients_z_boundaries.json")

# Wczytanie lub utworzenie podziału pacjentów (train: 70%, val: 15%, test: 15%)
splits_json_path = ZIP_DIR / "data_splits.json"
splits = get_or_create_splits(folder_path=GLI_ROOT, json_path=splits_json_path)

print(f"Liczba pacjentów:")
print(f" - Train:      {len(splits['train'])}")
print(f" - Validation: {len(splits['validate'])}")
print(f" - Test:       {len(splits['test'])}")

## 3. Test na 1 pacjencie (Sanity Check i Wizualizacja)

In [ ]:
import matplotlib.pyplot as plt

# Wybieramy pierwszego pacjenta z grupy treningowej
test_patient_id = splits["train"][0]
test_patient_dir = GLI_ROOT / test_patient_id
print(f"Testujemy pacjenta: {test_patient_id}")

# Ekstrakcja wycinków
slices = extract_patient_slices(test_patient_dir, config=config)
print(f"Wyekstrahowano {len(slices)} wycinków.")

# Zapis do folderu testowego
test_output_dir = Path("/content/test_processed")
saved_files = save_patients_to_pt(slices, test_output_dir)
print(f"Zapisano {len(saved_files)} plików .pt w {test_output_dir}")

# Odczytujemy środkowy wycinek (gdzie zazwyczaj widoczny jest guz)
sample_file = saved_files[len(saved_files) // 2]
data = torch.load(sample_file, weights_only=True)

img = data["image"]  # (4, 224, 224)
mask = data["mask"]  # (224, 224)

print("\n--- Weryfikacja danych tensora ---")
print(f"Kształt obrazu:       {img.shape}, typ: {img.dtype}")
print(f"Kształt maski:        {mask.shape}, typ: {mask.dtype}")
print(f"Unikalne klasy maski: {torch.unique(mask).tolist()}")
print(f"Wartości tła (naroże): {img[:, 0, 0].tolist()}")

# Wizualizacja 4 modalności MRI + Maska
modalities = ["T1", "T1CE", "T2", "FLAIR"]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i in range(4):
    axes[i].imshow(img[i].numpy(), cmap="gray")
    axes[i].set_title(modalities[i])
    axes[i].axis("off")

axes[4].imshow(mask.numpy(), cmap="tab10", vmin=0, vmax=4)
axes[4].set_title(f"Maska (z={data['z_idx']})")
axes[4].axis("off")

plt.tight_layout()
plt.show()

## 4. Pełny eksport całego zbioru (Uruchom po upewnieniu się, że test działa)

Folder docelowy zostanie utworzony na Google Drive (`processed/`).

In [ ]:
# Ścieżka docelowa na Google Drive
PROCESSED_ROOT = ZIP_DIR / "processed"

# Rozpoczęcie eksportu dla train, validate, test
# (Możesz uruchomić poniższą linijkę, gdy będziesz gotowy wygenerować pełny zbiór)
# stats = export_dataset_split(
#     data_root=GLI_ROOT,
#     splits=splits,
#     output_root=PROCESSED_ROOT,
#     config=config
# )
# print("Podsumowanie eksportu:", stats)